# **Week 1**

In [23]:
import zipfile
import os

# Specify the path to the zip file
zip_file_path = '/content/archive (3).zip'

# Specify the extraction directory
extract_dir = '/content/sales_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Open the zip file in read mode and extract all its contents
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Contents extracted to: {extract_dir}")


Contents extracted to: /content/sales_data


In [24]:
import os

extract_dir = '/content/sales_data'

# List the contents of the extracted directory
files_in_dir = os.listdir(extract_dir)

print(f"Files in '{extract_dir}':\n{files_in_dir}")


Files in '/content/sales_data':
['9. Sales-Data-Analysis.csv']


# **Load Sales Data**

In [25]:
import pandas as pd
import os

# Construct the full path to the sales data CSV file
extract_dir = '/content/sales_data'
file_name = '9. Sales-Data-Analysis.csv'
file_path = os.path.join(extract_dir, file_name)

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
print("DataFrame Head:")
print(df.head())

# Print a concise summary of the DataFrame
print("\nDataFrame Info:")
df.info()


DataFrame Head:
   Order ID        Date             Product  Price  Quantity Purchase Type  
0     10452  07-11-2022               Fries   3.49    573.07       Online    
1     10453  07-11-2022           Beverages   2.95    745.76       Online    
2     10454  07-11-2022       Sides & Other   4.99    200.40     In-store    
3     10455  08-11-2022             Burgers  12.99    569.67     In-store    
4     10456  08-11-2022  Chicken Sandwiches   9.95    201.01     In-store    

  Payment Method             Manager    City  
0      Gift Card    Tom      Jackson  London  
1      Gift Card         Pablo Perez  Madrid  
2      Gift Card       Joao    Silva  Lisbon  
3    Credit Card       Walter Muller  Berlin  
4    Credit Card       Walter Muller  Berlin  

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        25

# **Clean and Format Datetime Index**

In [26]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df = df.set_index('Date')
df = df.sort_index()

# Check for duplicate dates in the index
duplicate_dates = df.index.duplicated(keep=False)
if duplicate_dates.any():
    print("Duplicate dates found in the index. Here are the rows with duplicate dates:")
    print(df[duplicate_dates])
else:
    print("No duplicate dates found in the index.")

print("\nDataFrame after setting Date as index and sorting:")
print(df.head())
print("\nDataFrame Info after conversion and indexing:")
df.info()


Duplicate dates found in the index. Here are the rows with duplicate dates:
            Order ID             Product  Price  Quantity Purchase Type  
Date                                                                      
2022-11-07     10452               Fries   3.49    573.07       Online    
2022-11-07     10453           Beverages   2.95    745.76       Online    
2022-11-07     10454       Sides & Other   4.99    200.40     In-store    
2022-11-08     10455             Burgers  12.99    569.67     In-store    
2022-11-08     10456  Chicken Sandwiches   9.95    201.01     In-store    
...              ...                 ...    ...       ...           ...   
2022-12-28     10706  Chicken Sandwiches   9.95    301.51   Drive-thru    
2022-12-29     10711  Chicken Sandwiches   9.95    281.41   Drive-thru    
2022-12-29     10712               Fries   3.49    630.37   Drive-thru    
2022-12-29     10710             Burgers  12.99    754.43   Drive-thru    
2022-12-29     10713     

In [27]:
# Replacement for cell 27 — compute per-transaction Sales, then aggregate by date correctly

# (Assumes `df` currently contains the transaction-level DataFrame with Date as DatetimeIndex)
# Make a copy to preserve transaction-level data for later per-product work
df_transactions = df.copy()

# 1) Compute Sales per transaction (correct revenue calculation)
df_transactions['Sales'] = df_transactions['Price'] * df_transactions['Quantity']

# 2) Aggregate transactions by date (sum Sales, sum Quantity, keep other columns as summary)
df_daily = df_transactions.groupby(df_transactions.index).agg({
    'Order ID': 'first',                       # keep one representative order id (or change to 'count' if preferred)
    'Product': lambda x: ', '.join(x.astype(str)),
    'Price': 'mean',                           # optional reference - do NOT use to compute Sales
    'Quantity': 'sum',                         # total quantity per day
    'Purchase Type': lambda x: ', '.join(x.astype(str)),
    'Payment Method': lambda x: ', '.join(x.astype(str)),
    'Manager': lambda x: ', '.join(x.astype(str)),
    'City': lambda x: ', '.join(x.astype(str)),
    'Sales': 'sum'                             # aggregate the per-row Sales (correct daily revenue)
})

# 3) Resample to daily frequency to ensure continuity and fill missing days for Sales with 0
df_daily = df_daily.resample('D').agg({
    'Sales': 'sum',
    'Order ID': lambda x: list(x.dropna().unique()),
    'Product': lambda x: ', '.join(x.dropna().unique()),
    'Price': 'mean',
    'Quantity': 'sum',
    'Purchase Type': lambda x: ', '.join(x.dropna().unique()),
    'Payment Method': lambda x: ', '.join(x.dropna().unique()),
    'Manager': lambda x: ', '.join(x.dropna().unique()),
    'City': lambda x: ', '.join(x.dropna().unique())
})

df_daily['Sales'] = df_daily['Sales'].fillna(0)

# 4) Verification: totals should match original transaction-level Sales sum
total_from_rows = df_transactions['Sales'].sum()
total_from_daily = df_daily['Sales'].sum()
print(f"Total revenue from transactions: {total_from_rows:.2f}")
print(f"Total revenue after daily aggregation: {total_from_daily:.2f}")

# 5) Check for missing dates (should be none after resample but we report if any)
full_date_range = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
missing_dates = full_date_range.difference(df_daily.index)

if not missing_dates.empty:
    print(f"Missing dates found in the index: {missing_dates}")
else:
    print("No missing dates found in the daily index — datetime index is continuous.")

# 6) Replace df with the daily DataFrame for subsequent cells (same variable name used later in notebook)
df = df_daily
print("\nDataFrame after correct aggregation (daily):")
print(df.head())


No missing dates found in the index, the datetime index is continuous.

DataFrame after handling duplicate dates, calculating sales, and ensuring daily continuity:
                  Sales Order ID  
Date                               
2022-11-07   5788.26630  [10452]   
2022-11-08  12129.29825  [10455]   
2022-11-09  15168.99328  [10460]   
2022-11-10  14736.42040  [10467]   
2022-11-11  15562.87348  [10470]   

                                                      Product  Price  
Date                                                                   
2022-11-07                    Fries, Beverages, Sides & Other  3.810   
2022-11-08  Burgers, Chicken Sandwiches, Fries, Sides & Other  7.855   
2022-11-09  Burgers, Chicken Sandwiches, Fries, Beverages,...  6.874   
2022-11-10      Fries, Beverages, Burgers, Chicken Sandwiches  7.345   
2022-11-11  Burgers, Chicken Sandwiches, Fries, Beverages,...  6.874   


**Check for Missing Values and Outliers**

In [28]:
print("Missing values before handling (if any):")
print(df.isnull().sum())


Missing values before handling (if any):
Sales             0
Order ID          0
Product           0
Price             0
Quantity          0
Purchase Type     0
Payment Method    0
Manager           0
City              0
dtype: int64


# **Plot Overall Sales Trend**

In [29]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generate a box plot of the 'Sales' column to visually identify potential outliers
plt.figure(figsize=(10, 6))
sns.boxplot(y=df['Sales'])
plt.title('Box Plot of Sales to Identify Outliers')
plt.ylabel('Sales')
plt.grid(True)
plt.show()


# **Decompose Time Series (Weekly Seasonality)**

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a line plot of the 'Sales' column over time
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Sales'], label='Daily Sales', color='blue')

# Add title and labels
plt.title('Overall Sales Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')

# Add legend and grid
plt.legend()
plt.grid(True)

# Display the plot
plt.show()


In [31]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Apply seasonal_decompose with an additive model and a weekly period (7 days)
decomposition = seasonal_decompose(df['Sales'], model='additive', period=7)

# Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Time Series Decomposition (Weekly Seasonality)', y=1.02)

# Customize titles for better clarity
fig.axes[0].set_title('Original Sales')
fig.axes[1].set_title('Trend Component')
fig.axes[2].set_title('Seasonal Component (Weekly)')
fig.axes[3].set_title('Residual Component')

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()


In [32]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Apply seasonal_decompose with an additive model and an adjusted monthly period (26 days)
# Original request was for 30 days, but data length (53 observations) requires 2*period <= 53.
# 26 days is chosen as the largest approximate monthly period satisfying this (2*26 = 52).
decomposition = seasonal_decompose(df['Sales'], model='additive', period=26)

# Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Time Series Decomposition (Approx. Monthly Seasonality - 26 Days)', y=1.02)

# Customize titles for better clarity
fig.axes[0].set_title('Original Sales')
fig.axes[1].set_title('Trend Component')
fig.axes[2].set_title('Seasonal Component (Approx. Monthly)')
fig.axes[3].set_title('Residual Component')

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()


# **Analyze Autocorrelation**

In [33]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Determine an appropriate number of lags based on the data length
# It's generally recommended to have lags less than N/2, where N is the number of observations.
# For safety, let's use min(len(df) - 1, 20) to ensure we don't exceed data length and keep plots readable.
lags_to_consider = min(len(df) - 1, 20)

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot ACF
plot_acf(df['Sales'], ax=axes[0], lags=lags_to_consider)
axes[0].set_title('Autocorrelation Function (ACF) of Sales')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('Autocorrelation')
axes[0].grid(True)

# Plot PACF
plot_pacf(df['Sales'], ax=axes[1], lags=lags_to_consider)
axes[1].set_title('Partial Autocorrelation Function (PACF) of Sales')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('Partial Autocorrelation')
axes[1].grid(True)

plt.tight_layout()
plt.show()


# **Week 2: Advanced Feature Engineering**

In this section we create the feature-engineering pipeline required for modeling in Week 3. The cells below: (1) re-load the original CSV to produce independent cleaned outputs, (2) compute aggregate daily sales and per-product daily aggregates, (3) build time-based features, lags and rolling statistics, (4) create product-level features for top-N SKUs, and (5) save CSVs to /content/cleaned/. Run these cells in order.

In [ ]:
# Week 2 - A: Re-load CSV, compute per-transaction Sales, create daily and product-level aggregates
import pandas as pd
import os

file_path = '/content/sales_data/9. Sales-Data-Analysis.csv'
df_raw = pd.read_csv(file_path)

# parse date and set index
df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='%d-%m-%Y')
df_raw = df_raw.sort_values('Date').reset_index(drop=True)
df_raw = df_raw.set_index('Date')

# compute per-row Sales correctly
df_raw['Sales'] = df_raw['Price'] * df_raw['Quantity']

# 1) Daily total revenue series
daily_sales = df_raw['Sales'].resample('D').sum().fillna(0)
daily_sales = daily_sales.asfreq('D')

# 2) Product-level daily demand (quantity) and revenue
product_daily = df_raw.groupby([pd.Grouper(freq='D'), 'Product']).agg({
    'Quantity': 'sum',
    'Sales': 'sum'
}).reset_index().sort_values(['Product','Date'])

# create cleaned folder and save
os.makedirs('/content/cleaned', exist_ok=True)
daily_sales.to_frame('Sales').to_csv('/content/cleaned/cleaned_daily_sales.csv')
product_daily.to_csv('/content/cleaned/cleaned_product_daily.csv', index=False)

print('Saved /content/cleaned/cleaned_daily_sales.csv and cleaned_product_daily.csv')


In [ ]:
# Week 2 - B: Feature engineering function for aggregate series and saving features_daily.csv
import numpy as np

def make_time_features(df_series, country_holidays=None):
    df = df_series.to_frame('Sales').copy()
    df = df.asfreq('D')
    df['date'] = df.index
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
    df['is_month_start'] = df.index.is_month_start.astype(int)
    df['is_month_end'] = df.index.is_month_end.astype(int)
    if country_holidays:
        try:
            import holidays
            country_hols = holidays.CountryHoliday(country_holidays)
            df['is_holiday'] = df.index.to_series().apply(lambda d: 1 if d in country_hols else 0).astype(int)
        except Exception as e:
            print('holidays library missing or failed:', e)
            df['is_holiday'] = 0
    else:
        df['is_holiday'] = 0
    lags = [1,2,3,7,14,28]
    for lag in lags:
        df[f'lag_{lag}'] = df['Sales'].shift(lag)
    windows = [7,14,28]
    for w in windows:
        df[f'roll_mean_{w}'] = df['Sales'].rolling(window=w, min_periods=1).mean().shift(1)
        df[f'roll_std_{w}'] = df['Sales'].rolling(window=w, min_periods=1).std().shift(1).fillna(0)
    df['pct_change_1'] = df['Sales'].pct_change(1).fillna(0)
    df['pct_change_7'] = df['Sales'].pct_change(7).fillna(0)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['missing_after_shift'] = df[[f'lag_{l}' for l in lags]].isnull().any(axis=1).astype(int)
    df_feat = df.dropna(subset=[f'lag_{l}' for l in [1,7]])
    return df_feat

# load cleaned daily sales and create features
daily = pd.read_csv('/content/cleaned/cleaned_daily_sales.csv', index_col=0, parse_dates=True).squeeze('columns')
features_daily = make_time_features(daily, country_holidays=None)
features_daily.to_csv('/content/cleaned/features_daily.csv')
print('Saved /content/cleaned/features_daily.csv with shape', features_daily.shape)


In [ ]:
# Week 2 - C: Product-level features for top-N SKUs and saving features_product_topN.csv
product_daily = pd.read_csv('/content/cleaned/cleaned_product_daily.csv', parse_dates=['Date'])
agg_product_totals = product_daily.groupby('Product').agg({'Quantity':'sum','Sales':'sum'}).reset_index()
top_n = 10
top_products = agg_product_totals.sort_values('Quantity', ascending=False).head(top_n)['Product'].tolist()
print('Top products:', top_products)
product_features_list = []
min_history = 30
for prod in top_products:
    tmp = product_daily[product_daily['Product']==prod].set_index('Date').sort_index()
    idx = pd.date_range(tmp.index.min(), tmp.index.max(), freq='D')
    tmp = tmp.reindex(idx, fill_value=0)
    tmp.index.name = 'Date'
    series_qty = tmp['Quantity'].astype(float)
    feats = make_time_features(series_qty, country_holidays=None)
    feats['Product'] = prod
    if len(series_qty.dropna()) < min_history:
        print(f'skipping {prod} due to short history ({len(series_qty.dropna())} days)')
        continue
    product_features_list.append(feats.reset_index())

if product_features_list:
    product_features = pd.concat(product_features_list, ignore_index=True)
    product_features.to_csv('/content/cleaned/features_product_topN.csv', index=False)
    print('Product features saved:', product_features.shape)
else:
    print('No product features created. Check top_products or min_history.')


In [ ]:
# Week 2 - D: Time-based sequential split and rolling-origin CV helpers
from sklearn.model_selection import TimeSeriesSplit

def time_train_test_split(df, test_horizon=14):
    if 'Date' in df.columns:
        df = df.set_index('Date')
    train = df.iloc[:-test_horizon]
    test = df.iloc[-test_horizon:]
    return train, test

# Example usage for aggregate features (if features_daily exists)
import pandas as pd
features_daily = pd.read_csv('/content/cleaned/features_daily.csv', index_col=0, parse_dates=True)
train_df, test_df = time_train_test_split(features_daily, test_horizon=14)
print('train size, test size:', len(train_df), len(test_df))

# Rolling-origin CV example
tscv = TimeSeriesSplit(n_splits=3)
for fold, (tr_idx, val_idx) in enumerate(tscv.split(features_daily)):
    print('fold', fold, 'train end idx', tr_idx[-1], 'val end idx', val_idx[-1])


## Week 2 Results & Next Steps

The Week 2 cells create and save the following files under /content/cleaned/:
- cleaned_daily_sales.csv — aggregate daily revenue series (Sales)
- cleaned_product_daily.csv — per-product daily Quantity and Sales
- features_daily.csv — aggregate-level features (lags, rolling stats, date features)
- features_product_topN.csv — product-level features for top-N SKUs (Quantity target)

Next steps (Week 3 prep):
1) Re-run the notebook cells to execute Week 2 code and generate the CSVs.
2) Use features_daily.csv to train baseline models (Holt-Winters, SARIMAX) and compare metrics (MAE, RMSE).
3) Use features_product_topN.csv to produce per-SKU forecasts and compute inventory reorder points (safety stock and reorder point formulas are in the Week 4 plan).

If you'd like, I can (A) run baseline models on the cleaned aggregate series and produce evaluation plots, or (B) prepare inventory examples for top-5 products using forecasted Quantity — tell me which and I'll add the cells.